# Homework 3: Futures Data

## Question (a)
### Part (i)

In [16]:
library(rkdb)
# Let's try the direct IP address since hfm.princeton.edu refused us earlier
# db <- open_connection('52.156.9.80', 6007)
db <- open_connection('hfm.princeton.edu', 6007)
print("Connected!")

[1] "Connected!"


In [17]:
# 1. Define the KDB+/q query as a string
# We sum 'siz' for total volume and avg 'siz' for the average trade size, grouped by 'sym'
q_query <- "select total_vol: sum siz, avg_trade_size: avg siz by sym from trade where date within 2024.02.05 2024.02.09"

# 2. Send the query to the server and store the result
trade_stats <- execute(db, q_query)

# 3. KDB returns the 'sym' column as rownames by default in R. Let's make it a proper column.
trade_stats$sym <- rownames(trade_stats)
rownames(trade_stats) <- NULL

# 4. Avoid LaTeX table output in notebooks to prevent KaTeX parse errors
options(jupyter.display_mimetypes = c("text/plain", "text/html", "text/markdown"))

# 5. View the top results sorted by total volume (to see the most active symbols)
# R's 'order' function works similarly to pandas.sort_values() in Python
trade_stats_sorted <- trade_stats[order(-trade_stats$total_vol), ]
head(trade_stats_sorted, 10)

,sym,total_vol,avg_trade_size
,<chr>,<int64>,<dbl>
1,1,5,5.000000
2,2,5,5.000000
3,3,5,5.000000
4,4,5,5.000000
5,5,5,5.000000
6,6,19,3.800000
7,7,1,1.000000
8,8,75751,1.382091
9,9,23073,1.306660


In [18]:
# Try using the original db connection from the first cell
tryCatch(
  {
    # Define our q query string using a lambda function {} to keep things clean.
    # We join the trades and quotes, calculate (bsiz + asiz), and take the average.
    q_query_aii <- "{[syms]
        // 1. Pull the trades. We need sym and time to act as our 'lookup' keys.
        t_tbl: select sym, time from trade where date within 2024.02.05 2024.02.09, sym in syms;
        
        // 2. Pull the quotes. We need sym, time, and the quote sizes.
        q_tbl: select sym, time, bsiz, asiz from quote where date within 2024.02.05 2024.02.09, sym in syms;
        
        // 3. The As-Of Join. For every trade in t_tbl, find the prevailing quote in q_tbl based on sym and time.
        joined_tbl: aj[`sym`time; t_tbl; q_tbl];
        
        // 4. Calculate the average of the total quote size at those trade times
        select avg_quote_size: avg (bsiz + asiz) by sym from joined_tbl
    }[`ESH4`ZNH4`CLH4]" 
    
    # Execute the query on the server using the original db connection
    quote_sizes <<- execute(db, q_query_aii)
    
    # Clean up the output in R
    quote_sizes$sym <- rownames(quote_sizes)
    rownames(quote_sizes) <- NULL
    print(quote_sizes)
  },
  error = function(e) {
    print(paste("Part (ii) query failed:", e$message))
    # Create empty placeholder so later cells don't crash
    quote_sizes <<- data.frame(sym = character(), avg_quote_size = numeric())
  }
)

[1] "Part (ii) query failed: Not connected to kdb+ server."


In [19]:
tryCatch(
  {
    # 1. Define the query for part a(iii)
    q_query_aiii <- "{[syms]
        // Pull trades and quotes
        t_tbl: select sym, time from trade where date within 2024.02.05 2024.02.09, sym in syms;
        q_tbl: select sym, time, bid, ask from quote where date within 2024.02.05 2024.02.09, sym in syms;
        
        // As-of join to get prevailing bid/ask at exact trade times
        joined_tbl: aj[`sym`time; t_tbl; q_tbl];
        
        // Map the symbol to its instrument class (e.g., ESH4 -> ES)
        joined_tbl: update inst: sym2inst sym from joined_tbl;
        
        // Look up the tick size from the instinfo table
        tick_dict: exec minpxincr by inst from instinfo;
        joined_tbl: update tick: tick_dict inst from joined_tbl;
        
        // Calculate the fraction of trades where the spread is strictly greater than 1 tick
        // In q, boolean True/False averages out to the exact fraction!
        select frac_wide: avg (ask - bid) > tick by sym from joined_tbl
    }[`ESH4`ZNH4`CLH4]" 
    
    # 2. Execute the query
    wide_spreads <<- execute(db, q_query_aiii)
    
    # 3. Clean the R data frame
    wide_spreads$sym <- rownames(wide_spreads)
    rownames(wide_spreads) <- NULL
    print(wide_spreads)
  },
  error = function(e) {
    print(paste("Part (iii) query failed:", e$message))
    # Create empty placeholder so later cells don't crash
    wide_spreads <<- data.frame(sym = character(), frac_wide = numeric())
  }
)

[1] "Part (iii) query failed: Not connected to kdb+ server."


In [20]:
tryCatch(
  {
    # 1. Merge the results from a(i), a(ii), and a(iii) into one master data frame
    if (nrow(quote_sizes) > 0 && nrow(wide_spreads) > 0) {
      plot_data <- merge(trade_stats, quote_sizes, by="sym")
      plot_data <- merge(plot_data, wide_spreads, by="sym")
      
      # 2. Calculate the X-axis variable: Average quote size divided by average trade size
      plot_data$quote_trade_ratio <- plot_data$avg_quote_size / plot_data$avg_trade_size
      
      # 3. Create the scatter plot (Log scale on both axes)
      plot(plot_data$quote_trade_ratio, plot_data$frac_wide, 
           log = "xy", 
           pch = 19,               # Use solid dots
           col = "darkblue",
           xlab = "Average quote size / average trade size",
           ylab = "Fraction of trades when spread > 1 tick",
           main = "Part A: Spectrum of Tick Sizes",
           xlim = c(2, 500),       # Setting bounds based on Professor Almgren's chart
           ylim = c(0.002, 1))
      
      # 4. Add the symbol labels slightly above the dots
      text(plot_data$quote_trade_ratio, plot_data$frac_wide, 
           labels = plot_data$sym, 
           pos = 3,                # 'pos=3' puts the text above the point
           cex = 0.8)              # Make the text slightly smaller
    } else {
      print("Cannot create plot: missing data from earlier queries (quote_sizes or wide_spreads is empty)")
    }
  },
  error = function(e) {
    print(paste("Plot creation failed:", e$message))
  }
)

[1] "Cannot create plot: missing data from earlier queries (quote_sizes or wide_spreads is empty)"
